# Econometrics Lab 7 — Ridge and Lasso Regression

**Sorbonne Data Analytics**

**Question of the day:** we have 30 candidate predictors, only 100 observations, and the predictors are correlated. Can we predict better than OLS, and find the variables that really matter?

| Exercise | Slides |
|---|---|
| 1. Standardization and the OLS benchmark | 3–4, 6 |
| 2. Ridge by hand | 7–8, 12 |
| 3. Ridge: shrinkage and the bias–variance trade-off | 4, 9–10, 21 |
| 4. Lasso: shrinkage **and** selection | 13–16 |
| 5. Choosing $\lambda$ by cross-validation | 19, 22–23 |
| 6. Post-Lasso OLS | 25, 27 |

**Python reminder** (scikit-learn)

| Code | Meaning |
|---|---|
| `StandardScaler().fit(X)` then `.transform(X)` | standardize with the means and standard deviations of the data given to `fit` |
| `Ridge(alpha=lam).fit(X, y)` | ridge with penalty $\lambda$ = `lam` |
| `Lasso(alpha=a).fit(X, y)` | lasso. **Careful:** scikit-learn minimizes $\frac{1}{2n} RSS + \alpha \sum_j \lvert\beta_j\rvert$, so its $\alpha$ is the $\lambda$ of the slides divided by $2n$ |
| `.coef_`, `.intercept_`, `.predict(X)` | slopes, intercept, predictions |
| `mean_squared_error(y, y_pred)` | $\frac{1}{n}\sum_i (y_i - \hat y_i)^2$ |
| `np.eye(p)` | identity matrix $I_p$ |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import ElasticNetCV, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

## The data (simulated, so we know the truth)

- A **training** sample of $n = 100$ observations (to estimate) and a **test** sample of 1 000 observations (to measure the out-of-sample prediction error).
- $p = 30$ predictors with $Corr(x_j, x_k) = 0.7^{|j - k|}$: neighbouring predictors are strongly correlated.
- Only **5** predictors matter:
$$y = 5 + 3x_1 - 2x_2 + 1.5x_3 + x_{10} - x_{20} + u, \qquad u \sim N(0, 3^2)$$

The best possible test MSE is $V(u) = 9$. The predictors have variance 1, so after standardization the true coefficients are (almost) unchanged.

In [ ]:
n_train, n_test, p = 100, 1000, 30
names = np.array([f"x{j}" for j in range(1, p + 1)])

# Correlated predictors: Corr(x_j, x_k) = 0.7^|j - k|
corr = 0.7 ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))
X_all = rng.multivariate_normal(np.zeros(p), corr, size=n_train + n_test)

beta_true = np.zeros(p)
beta_true[[0, 1, 2, 9, 19]] = [3, -2, 1.5, 1, -1]  # x1, x2, x3, x10, x20
y_all = 5 + X_all @ beta_true + rng.normal(0, 3, n_train + n_test)

X_train, X_test = X_all[:n_train], X_all[n_train:]
y_train, y_test = y_all[:n_train], y_all[n_train:]
print(X_train.shape, X_test.shape)

---
## Exercise 1 — Standardization and the OLS benchmark (slides 3–4, 6)

1. Standardize the predictors with the means and standard deviations of the **training** sample, and apply the same transformation to the test sample. Why must we standardize before a penalized regression? Why not use the test sample to compute the means and standard deviations?
2. Estimate OLS on the training sample. Compute the MSE on the training and on the test samples, and compare them with the best possible value 9.
3. How many of the 25 irrelevant variables are "significant" at 5 %?

In [ ]:
scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)
Xs_test = ...

ols = ...  # sm.OLS on the standardized training data, with a constant
mse_ols_train = mean_squared_error(y_train, ols.predict(sm.add_constant(Xs_train)))
mse_ols_test = ...
print(f"OLS: train MSE = {mse_ols_train:.2f},  test MSE = {mse_ols_test:.2f}")

irrelevant = beta_true == 0
print(
    "Irrelevant variables significant at 5%:", ...
)  # hint: ols.pvalues[1:] skips the constant

**Your answer:**

---
## Exercise 2 — Ridge by hand (slides 7–8, 12)

$$\hat\beta^{Ridge} = (X'X + \lambda I)^{-1} X'y$$

The intercept is not penalized: with standardized $X$ we use the **centered** $y$, and the intercept is $\bar y$.

1. Compute $\hat\beta^{Ridge}$ for $\lambda = 10$ by hand and check it against `Ridge(alpha=10)`.
2. Check that $\lambda = 0$ gives back OLS.
3. Compare the smallest eigenvalue of $X'X$ and of $X'X + \lambda I$ (slide 12). Why does this make the estimator more stable?

In [ ]:
def ridge_by_hand(X, y, lam):
    y_centered = y - y.mean()
    I = np.eye(X.shape[1])
    return ...  # (X'X + lam I)^(-1) X' y_centered


b_ridge = ridge_by_hand(Xs_train, y_train, 10)
b_sklearn = Ridge(alpha=10).fit(Xs_train, y_train).coef_
print("Same as scikit-learn?", np.allclose(b_ridge, b_sklearn))
print(
    "lambda = 0 gives OLS?",
    np.allclose(ridge_by_hand(Xs_train, y_train, 0), ols.params[1:]),
)
print(
    pd.DataFrame(
        {"true": beta_true, "OLS": ols.params[1:], "Ridge": b_ridge}, index=names
    )
    .head(10)
    .round(2)
)

eigenvalues = np.linalg.eigvalsh(Xs_train.T @ Xs_train)
print("Smallest eigenvalue of X'X:", ..., "   of X'X + 10 I:", ...)

**Your answer:**

---
## Exercise 3 — Shrinkage and the bias–variance trade-off (slides 4, 9–10, 21)

For a grid of $\lambda$ from 0.01 to 10 000, fit a ridge on the training sample and store the coefficients, the train MSE and the test MSE.

1. Left plot: the coefficient paths. Do the coefficients become exactly zero?
2. Right plot: train and test MSE as functions of $\lambda$. Explain the two shapes with the bias–variance trade-off. Which $\lambda$ gives the best test MSE? Compare with OLS.

In [ ]:
lambdas = np.logspace(-2, 4, 50)  # 50 values from 0.01 to 10 000
coefs, mse_train, mse_test = [], [], []
for lam in lambdas:
    model = ...  # Ridge(alpha=lam) fitted on the training data
    coefs.append(model.coef_)
    mse_train.append(...)
    mse_test.append(...)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(lambdas, coefs)
axes[0].set(
    xscale="log", xlabel="lambda", ylabel="coefficient", title="Ridge coefficient paths"
)
axes[1].plot(lambdas, mse_train, label="train")
axes[1].plot(lambdas, mse_test, label="test")
axes[1].axhline(9, color="black", linestyle=":", label="best possible (9)")
axes[1].set(
    xscale="log", xlabel="lambda", ylabel="MSE", title="Bias-variance trade-off"
)
axes[1].legend()
plt.show()

best = np.argmin(mse_test)
print(
    f"Best lambda on the test sample: {lambdas[best]:.1f}, test MSE = {mse_test[best]:.2f} (OLS: {mse_ols_test:.2f})"
)

**Your answer:**

---
## Exercise 4 — Lasso: shrinkage **and** selection (slides 13–16)

1. Fit a lasso for a grid of $\alpha$ (scikit-learn's scale, see the reminder) and plot the coefficient paths and the number of non-zero coefficients. What is the difference with ridge?
2. Fit a lasso with $\alpha = 0.5$. Which variables are selected? Compare with the 5 true ones.

In [ ]:
alphas = np.logspace(-3, 1, 50)
coefs, n_nonzero = [], []
for a in alphas:
    model = ...  # Lasso(alpha=a) fitted on the training data
    coefs.append(model.coef_)
    n_nonzero.append(...)  # number of coefficients different from 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(alphas, coefs)
axes[0].set(
    xscale="log", xlabel="alpha", ylabel="coefficient", title="Lasso coefficient paths"
)
axes[1].plot(alphas, n_nonzero)
axes[1].set(xscale="log", xlabel="alpha", ylabel="number of non-zero coefficients")
plt.show()

lasso = ...  # Lasso(alpha=0.5)
print("Selected:", names[lasso.coef_ != 0])
print("True:    ", names[beta_true != 0])

**Your answer:**

---
## Exercise 5 — Choosing $\lambda$ by cross-validation (slides 19, 22–23)

In real life there is no test sample to choose $\lambda$ (choosing it on the test sample, as in Exercise 3, is cheating). We use **5-fold cross-validation on the training sample only**.

1. Fit `RidgeCV`, `LassoCV` and `ElasticNetCV` with 5 folds. For the elastic net, `l1_ratio=0.5` is the mixing parameter (the $\alpha$ of slide 19; scikit-learn's `alpha` is the overall penalty).
2. Apply the **one-standard-error rule** to the lasso: take the largest $\alpha$ whose CV error is below the minimum CV error plus one standard error.
3. Compare all models on the test sample: test MSE and number of non-zero coefficients. Which model would you choose?

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 100), cv=5).fit(Xs_train, y_train)
lasso_cv = LassoCV(cv=5).fit(Xs_train, y_train)
enet_cv = ElasticNetCV(l1_ratio=0.5, cv=5).fit(Xs_train, y_train)

# One-standard-error rule. lasso_cv.mse_path_ has one row per alpha and one column per fold
cv_mean = lasso_cv.mse_path_.mean(axis=1)
cv_se = lasso_cv.mse_path_.std(axis=1) / np.sqrt(5)
i_min = np.argmin(cv_mean)
alpha_1se = (
    ...
)  # largest alpha in lasso_cv.alphas_ with cv_mean <= cv_mean[i_min] + cv_se[i_min]
lasso_1se = Lasso(alpha=alpha_1se).fit(Xs_train, y_train)
print(f"alpha min = {lasso_cv.alpha_:.3f},  alpha 1se = {alpha_1se:.3f}")

results = {"OLS": [mse_ols_test, p]}
models = {
    "Ridge (CV)": ridge_cv,
    "Lasso (CV min)": lasso_cv,
    "Lasso (1 s.e.)": lasso_1se,
    "Elastic net (CV)": enet_cv,
}
for name, model in models.items():
    results[name] = [..., ...]  # test MSE, number of non-zero coefficients
print(
    pd.DataFrame.from_dict(
        results, orient="index", columns=["test MSE", "non-zero coefs"]
    ).round(2)
)

**Your answer:**

---
## Exercise 6 — Post-Lasso OLS (slides 25, 27)

The lasso shrinks the coefficients of the selected variables towards zero. **Post-Lasso:** re-estimate by OLS using only the variables selected by `lasso_cv`.

1. Compare the true coefficients, the lasso coefficients and the post-Lasso OLS coefficients of the selected variables.
2. Compute the test MSE of post-Lasso OLS.
3. Can we trust the p-values of this OLS regression? Why?

In [ ]:
selected = lasso_cv.coef_ != 0
post = ...  # OLS of y_train on a constant and Xs_train[:, selected]
mse_post = ...  # test MSE: use Xs_test[:, selected]

print(
    pd.DataFrame(
        {
            "true": beta_true[selected],
            "Lasso": lasso_cv.coef_[selected],
            "Post-Lasso OLS": post.params[1:],
        },
        index=names[selected],
    ).round(2)
)
print(f"Post-Lasso test MSE = {mse_post:.2f}")

**Your answer:**

---
## Summary

| | OLS | Ridge | Lasso |
|---|---|---|---|
| Penalty | none | $\lambda \sum_j \beta_j^2$ | $\lambda \sum_j \lvert\beta_j\rvert$ |
| Closed form | $(X'X)^{-1}X'y$ | $(X'X + \lambda I)^{-1}X'y$ | no (iterative) |
| Coefficients exactly 0 | no | no | yes: selection |
| Bias / variance | unbiased, high variance | biased, lower variance | biased, lower variance |
| Use it for | inference when $n \gg p$ | prediction, multicollinearity | prediction and selection |

**Practical rules:** standardize (with the training sample only), choose $\lambda$ by cross-validation, judge the models on data not used for estimation, never trust naive p-values after selection, and remember that regularization is a tool for **prediction**, not for causal inference.